In [1]:
import torch
import time
import copy
import os
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

if os.path.basename(os.getcwd()) == 'exercises':
    os.chdir('../')
    
from ctm.ctm import ContinuousThoughtMachine
from ctm.img_coder import MinesweeperConvEncoder
from ctm.action_head import UniversalActionHead
from ctm.critic_head import UniversalCriticHead
from ctm.ctm_rl import ContinuousThoughtMachineRL


/home/mwuerstle@corp.exxcellent.de/miniconda3/envs/agi-project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device_config = -1

# Configure device string (support MPS on macOS)
if device_config != -1:
    device = f'cuda:{device_config}'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Running model CTM on {device}')
torch.set_default_device(device)

Running model CTM on cpu


In [3]:
from environments.minesweeper.minesweeper import MinesweeperEnv

width = 6
height = 6
n_mines = 10

env = MinesweeperEnv(width, height, n_mines)

current_state = env.state_im
board = current_state.reshape(1, env.ntiles)
unsolved = [i for i, x in enumerate(board[0]) if x==-0.125]
move = np.random.choice(unsolved)
env.step(move)
env.draw_state(env.state_im)
print(current_state.shape)



/home/mwuerstle@corp.exxcellent.de/Uni/agi_project/environments/minesweeper/minesweeper.py:123: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(state_df.style.applymap(self.color_state))


,0,1,2,3,4,5
0,-1,-1,2,-1,-1,-1
1,-1,-1,-1,-1,-1,-1
2,-1,-1,-1,-1,-1,-1
3,-1,-1,-1,-1,-1,-1
4,-1,-1,-1,-1,-1,-1
5,-1,-1,-1,-1,-1,-1


(6, 6, 1)


In [4]:
ctm_latent_dim = 256

env.reset()

minesweeper_enc = MinesweeperConvEncoder(ctm_latent_dim, env.state_im.shape)
# Create Tensor with the correct shape for encoding 
state_tensor = torch.from_numpy(env.state_im.T).float().unsqueeze(0)
encoded_state = minesweeper_enc.forward(state_tensor)
print(encoded_state.shape)

torch.Size([1, 256])


In [5]:
d_input = 256

ctm = ContinuousThoughtMachine(iterations=5, 
                               d_model=2048, 
                               d_input=d_input, 
                               heads=8, 
                               n_synch_out=64, 
                               n_synch_action=32,
                               synapse_depth=8, 
                               memory_length=25, 
                               deep_nlms=False,
                               memory_hidden_dims=32, 
                               do_layernorm_nlm=False,
                               backbone_type='none',
                               positional_embedding_type='none',
                               out_dims=256
                               )

try:
    # Determine pseudo input shape based on dataset
    pseudo_inputs = torch.zeros((1, 1, d_input), device=device).float()
    ctm(pseudo_inputs)
except Exception as e:
        print(f"Warning: Pseudo forward pass failed: {e}")

latent_state = encoded_state.unsqueeze(0)
print(latent_state.shape)
predictions_raw, certainties, synchronisation = ctm(latent_state)
final_thought = predictions_raw[:, :, -1]
print(final_thought.shape)

Using neuron select type: random-pairing
Synch representation size action: 32
Synch representation size out: 64
torch.Size([1, 1, 256])
torch.Size([1, 256])


In [6]:
action_head = UniversalActionHead(latent_dim=256, num_actions=env.ntiles)
action_logits = action_head(final_thought)
action_probs = torch.softmax(action_logits, dim=-1)
action = torch.multinomial(action_probs, num_samples=1)
print(action.item())

5


In [7]:
critic_head = UniversalCriticHead(latent_dim=256)
critic = critic_head(final_thought)
print(critic.item())

0.08516155928373337


In [8]:
def get_action(self, state, env):
    board = state.reshape(1, env.ntiles)
    unsolved = [i for i, x in enumerate(board[0]) if x==-0.125]

    rand = np.random.random() # random value b/w 0 & 1

    if rand < self.epsilon: # random move (explore)
        move = np.random.choice(unsolved)
    else:
        moves = self.model.predict(np.reshape(state, (1, self.env.nrows, self.env.ncols, 1)))
        moves[board!=-0.125] = np.min(moves) # set already clicked tiles to min value
        move = np.argmax(moves)

    return move

In [48]:
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical

from ctm.action_head import UniversalActionHead
from ctm.critic_head import UniversalCriticHead

class CTMAgent(nn.Module):
    def __init__(self, ctm, continuous_state_trace, device):
        super().__init__()

        self.continious_state_trace = continuous_state_trace
        self.device = device


        self.ctm = ctm
        actor_input_dim = critic_input_dim = self.ctm.synch_representation_size_out
        print(actor_input_dim, critic_input_dim)

        self.actor = UniversalActionHead(latent_dim=actor_input_dim)
        self.critic = UniversalCriticHead(latent_dim=critic_input_dim)


    def get_initial_state(self, num_envs):
        return self.get_initial_ctm_state(num_envs)

    def get_initial_ctm_state(self, num_envs):
        initial_state_trace = torch.repeat_interleave(self.ctm.start_trace.unsqueeze(0), num_envs, 0)
        initial_activated_state_trace = torch.repeat_interleave(self.ctm.start_activated_trace.unsqueeze(0), num_envs, 0)
        return initial_state_trace, initial_activated_state_trace
    
    def get_initial_lstm_state(self, num_envs):
        initial_hidden_state = torch.repeat_interleave(self.ctm.start_hidden_state.unsqueeze(0), num_envs, 0)
        initial_cell_state = torch.repeat_interleave(self.ctm.start_cell_state.unsqueeze(0), num_envs, 0)
        return initial_hidden_state, initial_cell_state

    def _get_hidden_states(self, state, done, num_envs):
        return self._get_ctm_hidden_states(state, done, num_envs)

    def _get_ctm_hidden_states(self, ctm_state, done, num_envs):
        initial_state_trace, initial_activated_state_trace = self.get_initial_ctm_state(num_envs)
        if self.continious_state_trace:
            masked_previous_state_trace = (1.0 - done).view(-1, 1, 1) * ctm_state[0]
            masked_previous_activated_state_trace = (1.0 - done).view(-1, 1, 1) * ctm_state[1]
            masked_initial_state_trace = done.view(-1, 1, 1) * initial_state_trace
            masked_initial_activated_state_trace = done.view(-1, 1, 1) * initial_activated_state_trace
            return (masked_previous_state_trace + masked_initial_state_trace), (masked_previous_activated_state_trace + masked_initial_activated_state_trace)
        else:
            return (initial_state_trace, initial_activated_state_trace)

    def get_states(self, x, ctm_state, done, track=False):
        num_envs = ctm_state[0].shape[0]

        if len(x.shape) == 4:
            _, C, H, W = x.shape
            xs = x.reshape((-1, num_envs, C, H, W))
        elif len(x.shape) == 2:
            _, C = x.shape
            xs = x.reshape((-1, num_envs, C))
        else:
            raise ValueError("Input shape not supported.")
        
        done = done.reshape((-1, num_envs))
        new_hidden = []
        for x, d in zip(xs, done):
            if not track:
                synchronisation, ctm_state = self.ctm(x, self._get_hidden_states(ctm_state, d, num_envs))
                tracking_data = None
                new_hidden += [synchronisation]
            else:
                synchronisation, ctm_state, pre_activations, post_activations = self.ctm(x, self._get_hidden_states(ctm_state, d, num_envs), track=True)
                tracking_data = {
                    'pre_activations': pre_activations,
                    'post_activations': post_activations,
                    'synchronisation': synchronisation.detach().cpu().numpy(),
                }
                new_hidden += [synchronisation]
        
        return torch.cat(new_hidden), ctm_state, tracking_data

    def get_value(self, x, ctm_state, done):
        hidden, _, _ = self.get_states(x, ctm_state, done)
        return self.critic(hidden)
    
    def get_action_and_value(self, x, ctm_state, done, action=None, track=False):
        hidden, ctm_state, tracking_data = self.get_states(x, ctm_state, done, track=track)
        action_logits = self.actor(hidden)
        action_probs = Categorical(logits=action_logits)

        if action is None:
            action = action_probs.sample()
        
        value = self.critic(hidden)
        
        return action, action_probs.log_prob(action), action_probs.entropy(), value, ctm_state, tracking_data, action_logits, action_probs.probs

In [ ]:
# training Inputs
num_steps = 10
num_envs = 1
num_minibatches = 1
batch_size = num_envs * num_steps
update_epochs = 1
# reward
discount_gamma = 0.99
gae_lambda = 0.95
# loss
norm_adv = True
ent_coef = 0.1
clip_vloss = False
clip_coef = 0.1
vf_coef = 0.25

d_input = 256

ctm = ContinuousThoughtMachineRL(iterations=5, 
                               d_model=2048, 
                               d_input=d_input, 
                               n_synch_out=64, 
                               synapse_depth=8, 
                               memory_length=25, 
                               deep_nlms=False,
                               memory_hidden_dims=32, 
                               do_layernorm_nlm=False,
                               backbone_type='minesweeper-backbone',
                               )

agent = CTMAgent(ctm=ctm, continuous_state_trace=True, device=device)

# optimizer = optim.Adam(agent.parameters(), lr=args.lr, eps=1e-5)

# init for tracking
obs = torch.zeros((num_steps, num_envs) + latent_state.shape).to(device)
actions = torch.zeros((num_steps, num_envs) + action.shape).to(device)
logprobs = torch.zeros((num_steps, num_envs)).to(device)
rewards = torch.zeros((num_steps, num_envs)).to(device)
dones = torch.zeros((num_steps, num_envs)).to(device)
values = torch.zeros((num_steps, num_envs)).to(device)

env.reset()
state_tensor = torch.from_numpy(env.state_im.T).float().unsqueeze(0)
next_obs = minesweeper_enc.forward(state_tensor)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(num_envs).to(device)
next_state = agent.get_initial_state(num_envs)

Using neuron select type: first-last
Synch representation size action: 0
Synch representation size out: 2080
2080 2080


In [ ]:
initial_state = (next_state[0].clone(), next_state[1].clone())

for step in range(0, num_steps):
    next_obs = torch.Tensor(next_obs).to(device)
    #global_step += args.num_envs
    with torch.no_grad():
        action, logprob, _, value, next_state, _, _, _ = agent.get_action_and_value(next_obs, next_state, next_done)
        values[step] = value.flatten()
    
    next_obs, reward, done = env.step(action)
    done = np.logical_or(done, done)
    
    # encode state after action
    next_obs = minesweeper_enc.forward(torch.from_numpy(next_obs.T).float().unsqueeze(0))
    next_obs, next_done, reward = torch.Tensor(next_obs).to(device), torch.Tensor([done]).to(device)

    # track everything
    obs[step] = next_obs
    dones[step] = next_done
    actions[step] = action
    logprobs[step] = logprob
    rewards[step] = torch.tensor(reward).to(device).view(-1)

In [52]:
# get next value
with torch.no_grad():
    next_value = agent.get_value(next_obs, next_state, next_done).reshape(1, -1)
    advantages = torch.zeros_like(rewards).to(device)
    lastgaelam = 0
    for t in reversed(range(num_steps)):
        if t == num_steps - 1:
            nextnonterminal = 1.0 - next_done
            nextvalues = next_value
        else:
            nextnonterminal = 1.0 - dones[t + 1]
            nextvalues = values[t + 1]
        delta = rewards[t] + discount_gamma * nextvalues * nextnonterminal - values[t]
        advantages[t] = lastgaelam = delta + discount_gamma * gae_lambda * nextnonterminal * lastgaelam
    returns = advantages + values

b_obs = obs.reshape((-1,) + latent_state.shape)
b_logprobs = logprobs.reshape(-1)
b_actions = actions.reshape((-1,) + action.shape)
b_dones = dones.reshape(-1)
b_advantages = advantages.reshape(-1)
b_returns = returns.reshape(-1)
b_values = values.reshape(-1)

assert num_envs % num_minibatches == 0
envsperbatch = num_envs // num_minibatches
envinds = np.arange(num_envs)
flatinds = np.arange(batch_size).reshape(num_steps, num_envs)
clipfracs = []

ValueError: cannot reshape array of size 1 into shape (10,1)

In [ ]:
for epoch in range(update_epochs):
    for start in range(0, num_envs, envsperbatch):
        end = start + envsperbatch
        mbenvinds = envinds[start:end]
        mb_inds = flatinds[:, mbenvinds].ravel()

        selected_hidden_state = (initial_state[0][mbenvinds,:,:], initial_state[1][mbenvinds,:,:])


        _, newlogprob, entropy, newvalue, _, _, _, _ = agent.get_action_and_value(
            b_obs[mb_inds],
            selected_hidden_state,
            b_dones[mb_inds],
            b_actions.long()[mb_inds],
        )

        logratio = newlogprob - b_logprobs[mb_inds]
        ratio = logratio.exp()

        with torch.no_grad():
            old_approx_kl = (-logratio).mean()
            approx_kl = ((ratio - 1) - logratio).mean()
            clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]

        mb_advantages = b_advantages[mb_inds]
        if norm_adv:
            mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

        # Policy loss
        pg_loss1 = -mb_advantages * ratio
        pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
        pg_loss = torch.max(pg_loss1, pg_loss2).mean()

        # Value loss
        newvalue = newvalue.view(-1)
        if clip_vloss:
            v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
            v_clipped = b_values[mb_inds] + torch.clamp(
                newvalue - b_values[mb_inds],
                -clip_coef,
                clip_coef,
            )
            v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
            v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
            v_loss = 0.5 * v_loss_max.mean()
        else:
            v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()


        entropy_loss = entropy.mean()
        loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef

        optimizer.zero_grad()
        loss.backward()